# Training a Single Neuron

This experiment demonstrates the complete learning workflow in `nnlab`.

The model consists of:

- a dense layer providing a linear transformation,
- a parameterized activation function,
- a mean squared error loss,
- stochastic gradient descent optimization.

The network architecture is:

$$
x \rightarrow Wx+b \rightarrow \phi(x) \rightarrow \hat{y}
$$

The training process is:

$$
\text{prediction}
\rightarrow
\text{loss}
\rightarrow
\text{gradient}
\rightarrow
\text{backward pass}
\rightarrow
\text{parameter update}
$$

The goal is not to create a high-capacity model, but to observe how individual neural network components interact.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nnlab.kernels import LogisticKernel

from nnlab.activations import ParameterizedActivation

from nnlab.layers import (
    Dense,
    ActivationLayer,
)

from nnlab.models import FeedForward

from nnlab.losses import MeanSquaredError

from nnlab.optimizers import SGD

## Training Data

A single neuron with a logistic activation can naturally represent a
scaled and shifted sigmoid function.

Therefore, the first experiment uses a sigmoid target function.

The purpose is to verify the learning pipeline before moving to more
complex approximation problems.

In [ ]:
x = np.linspace( -5.0, 5.0, 200, ).reshape( -1, 1, )

target = 1.0 / (
    1.0 + np.exp(-x)
)

plt.figure(figsize=(8,4))

plt.plot(
    x,
    target,
)

plt.title(
    "Target Function"
)

plt.xlabel(
    "Input"
)

plt.ylabel(
    "Output"
)

plt.grid(True)

plt.show()

## Model Definition

The neuron contains:

1. A dense layer:

$$
z = Wx+b
$$

2. A logistic activation:

$$
\hat{y}=\phi(z)
$$

The dense layer parameters are initialized randomly and will be learned during training.

In [ ]:
kernel = LogisticKernel()

activation = ParameterizedActivation(
    kernel=kernel,
    center=0.0,
    kernel_scale=1.0,
    amplitude=1.0,
    bias=0.0,
)

dense = Dense(
    input_size=1,
    output_size=1,
)

activation_layer = ActivationLayer(
    activation=activation,
)

model = FeedForward(
    layers=[
        dense,
        activation_layer,
    ],
)

In [ ]:
prediction = model.forward(
    x,
)

plt.figure(figsize=(8,4))

plt.plot(
    x,
    target,
    label="Target",
)

plt.plot(
    x,
    prediction,
    label="Initial Prediction",
)

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
loss = MeanSquaredError()

optimizer = SGD(
    learning_rate=0.1,
)

## Training

Each epoch performs:

1. Forward pass:
   
$$
\hat{y}=f(x)
$$

2. Loss calculation:

$$
L=\frac{1}{n}\sum(\hat{y}-y)^2
$$

3. Backward pass:

$$
\frac{\partial L}{\partial \theta}
$$

4. Parameter update:

$$
\theta=\theta-\eta\nabla_\theta L
$$

In [ ]:
epochs = 500

history = []

for epoch in range(epochs):

    prediction = model.forward(
        x,
    )

    error = loss.forward(
        prediction,
        target,
    )

    gradient = loss.derivative(
        prediction,
        target,
    )

    model.backward(
        gradient,
    )

    optimizer.step(
        model.parameters(),
        model.gradients(),
    )

    history.append(
        error,
    )

In [ ]:
plt.figure(figsize=(8,4))

plt.plot(
    history,
)

plt.title(
    "Training Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "MSE"
)

plt.grid(True)

plt.show()

In [ ]:
prediction = model.forward(
    x,
)

plt.figure(figsize=(8,4))

plt.plot(
    x,
    target,
    label="Target",
)

plt.plot(
    x,
    prediction,
    label="Learned Prediction",
)

plt.legend()

plt.title(
    "Single Neuron Approximation"
)

plt.xlabel(
    "Input"
)

plt.ylabel(
    "Output"
)

plt.grid(True)

plt.show()

In [ ]:
print(
    "Weight:",
    dense.weights,
)

print(
    "Bias:",
    dense.bias,
)

## Observations

This experiment demonstrates the complete neural network training cycle.

The model learned by repeatedly:

- evaluating its current approximation,
- measuring error,
- propagating error information backward,
- updating its parameters.

Although this model is extremely small, the same sequence forms the basis of larger neural networks.

Future experiments will extend this system by adding:

- multiple neurons,
- hidden layers,
- alternative activation functions,
- different optimization methods,
- more complex target functions.